In [2]:
import rich.console
from qibolab import (
    AcquisitionType,
    AveragingMode,
    Pulse,
    PulseSequence,
    Rectangular,
    create_platform,
    Custom
)
from qibolab._core.components.filters import ExponentialFilter, FiniteImpulseResponseFilter
import os
import numpy as np
from qblox_instruments.simulations.predistortions import get_filter_delay, exponential_overshoot_correction, fir_correction, get_impulse_response

In [2]:
os.environ["QIBOLAB_PLATFORMS"] = "/home/users/roy.stegeman/github/qibolab_platforms_qrc"

In [3]:
cons = rich.console.Console(color_system="truecolor")
headers = "b i color(8)"

In [4]:
platform = create_platform("qbc02")
ch = platform.qubits["D1"].flux

In [5]:
platform.disconnect()

In [6]:
platform.connect()

/nfs/users/roy.stegeman/calibration/venv_qw5q/lib/python3.10/site-packages/qcodes/instrument/instrument_base.py:642: UserWarning: Changed qw21q-d to qw21q_d for instrument identifier
  warnings.warn(f"Changed {name} to {new_name} for instrument identifier")


In [7]:
t = np.arange(0, 3000) 
A = 0.5
tau = 200
u = np.zeros(t.size)
u[1000:2000] = 1
s = (1 + A * np.exp(-(t-1000) / tau)) * u
s[2000:] = (- A * np.exp(-(t[:1000]) / tau))

In [8]:
envelope = Custom(i_=s, q_=np.zeros_like(s))

In [9]:
sequence = PulseSequence(
    [(ch, Pulse(amplitude=0.1, duration=3000, envelope=envelope))]
)

In [10]:
b = np.zeros(32)
b[:10] = 0.1
b[-10:] = -0.1

res = platform.execute(
    [sequence],
    updates=[{"D1/flux": {"filters": [ExponentialFilter(**{"amplitude": 0.5, "tau": 200})]}}],
    # updates=[{"D1/flux": {"filters": [FiniteImpulseResponseFilter(coefficients=b.tolist())]}}],
    # updates=[{"D1/flux": {"filters": [FiniteImpulseResponseFilter(coefficients=b.tolist()), ExponentialFilter(**{"amplitude": 0.5, "tau": 200})]}}],
    nshots=1e1,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
)

/nfs/users/roy.stegeman/github/qibolab/src/qibolab/_core/instruments/qblox/config/module.py:176: QCoDeSDeprecationWarning: Call set directly on the parameter.
  mod.set(config, value)
/nfs/users/roy.stegeman/github/qibolab/src/qibolab/_core/instruments/qblox/config/sequencer.py:148: QCoDeSDeprecationWarning: Call set directly on the parameter.
  seq.set(name, value)


In [12]:
cluster = platform.instruments["qblox"]
cluster.cluster.print_readable_snapshot(update=True)

qw21q_d:
	parameter                        value
--------------------------------------------------------------------------------
IDN                               :	{'manufacturer': 'qblox', 'model': 'clust...
ext_trigger_input_delay           :	0 (ps)
ext_trigger_input_trigger_address :	0 
ext_trigger_input_trigger_en      :	False 
led_brightness                    :	high 
reference_source                  :	internal 
trigger10_monitor_count           :	0 
trigger11_monitor_count           :	0 
trigger12_monitor_count           :	0 
trigger13_monitor_count           :	0 
trigger14_monitor_count           :	0 
trigger15_monitor_count           :	0 
trigger1_monitor_count            :	0 
trigger2_monitor_count            :	0 
trigger3_monitor_count            :	0 
trigger4_monitor_count            :	0 
trigger5_monitor_count            :	0 
trigger6_monitor_count            :	0 
trigger7_monitor_count            :	0 
trigger8_monitor_count            :	0 
trigger9_monitor_count        